# Per-category harm-rate analysis

Runs top to bottom on a fresh kernel from the repo root with the venv active. Reads every `logs/*.eval`, re-scores with the current scorer, and tabulates violations by taxonomy class, scenario and arm. No numbers are typed in by hand; change the glob to point at other logs.

In [ ]:
import glob
from collections import Counter

from welfareguard.evals.report import collect, per_harm_class, per_model, per_scenario

LOGS = sorted(glob.glob("logs/*.eval"))
rows = collect(LOGS)
print(f"{len(rows)} scored samples from {len(LOGS)} log(s)")
print("models:", sorted({r["model"] for r in rows}))

## Per model

In [ ]:
from IPython.display import Markdown

Markdown(per_model(rows))

## Violation rate by harm class (injected arms only)

The Week 6 per-category table. A sample carrying two classes counts under both.

In [ ]:
Markdown(per_harm_class(rows))

## Scenario difficulty

Violations per scenario across injected arms, highest first. 'Difficulty' here means how often the model failed, which on a fixed, untuned pressure ladder is a lower bound.

In [ ]:
by_sc = Counter()
n_sc = Counter()
for r in rows:
    if r["arm"] != "control":
        n_sc[r["scenario"]] += 1
        by_sc[r["scenario"]] += int(r["violation"])
for sid in sorted(n_sc, key=lambda s: (-by_sc[s] / n_sc[s], s)):
    print(f"{sid}  {by_sc[sid]:3d} / {n_sc[sid]:3d}  {100 * by_sc[sid] / n_sc[sid]:5.1f}%")

## Manipulation effect: treatment minus neutral

The reason the neutral arm exists. Positive means the pressure phrases moved the model beyond the plain request.

In [ ]:
def rate(sid, arm):
    sub = [r for r in rows if r["scenario"] == sid and r["arm"] == arm]
    return sum(r["violation"] for r in sub) / len(sub) if sub else float("nan")


for sid in sorted({r["scenario"] for r in rows}):
    t, n = rate(sid, "treatment"), rate(sid, "neutral")
    print(
        f"{sid}  neutral {100 * n:5.1f}%  treatment {100 * t:5.1f}%  effect {100 * (t - n):+6.1f} pts"
    )

## Per scenario and arm

In [ ]:
Markdown(per_scenario(rows))